<a href="https://colab.research.google.com/github/VladShajdulin/OTUS/blob/main/home_work_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install evaluate
! git clone https://github.com/RussianNLP/RuCoLA

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
Cloning into 'RuCoLA'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 76 (delta 31), reused 52 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 948.93 KiB | 8.18 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [20]:
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
import evaluate
from itertools import accumulate

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print('Device', device)

Device cuda


In [4]:
train_df = pd.read_csv('/content/RuCoLA/data/in_domain_train.csv')[['sentence', 'acceptable']]
test_df = pd.read_csv('/content/RuCoLA/data/in_domain_dev.csv')[['sentence', 'acceptable']]
train, val = train_test_split(
    train_df,
    test_size=0.2,
    random_state=345,
    shuffle=True,
    stratify=train_df['acceptable']
)
print(train['acceptable'].mean())
train_df

0.7451945988880063


,sentence,acceptable
0,"Вдруг решетка беззвучно поехала в сторону, и н...",1
1,Этим летом не никуда ездили.,0
2,Только Иван выразил какую бы то ни было готовн...,1
3,"Теперь ты видишь собственными глазами, как тут...",1
4,На поверку вся теория оказалась полной чепухой.,1
...,...,...
7864,Установки не было введено в действие.,0
7865,"Конечно, против такой системы ценностей решите...",0
7866,Симптомов болезни не исчезло.,0
7867,Послезавтра температура у больного снижается д...,0


In [5]:
clf_metrics = evaluate.combine(['f1', 'accuracy'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# 1. Обучение BERT

In [ ]:
name_bert = 'ai-forever/ruBert-base'
tokenizer = AutoTokenizer.from_pretrained(name_bert)
model = AutoModelForSequenceClassification.from_pretrained(name_bert, num_labels=2).to(device)

train, val = Dataset.from_pandas(train), Dataset.from_pandas(val)
train = train.map(lambda x: tokenizer(x['sentence']), batched=True).rename_column('acceptable', 'label')
val = val.map(lambda x: tokenizer(x['sentence']), batched=True).rename_column('acceptable', 'label')

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors='pt'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/716M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruBert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
  preds, labels = eval_pred

  return clf_metrics.compute(predictions=preds.argmax(axis=1), references=labels)

args = TrainingArguments(
    output_dir='bert_model',
    save_strategy='no',
    eval_strategy='steps',
    eval_steps=100,
    logging_strategy='steps',
    logging_steps=100,

    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    lr_scheduler_type='constant'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train,
    eval_dataset=val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss,Validation Loss,F1,Accuracy
100,0.563100,0.573240,0.854227,0.745870
200,0.529400,0.522377,0.860552,0.762389
300,0.503000,0.513935,0.871504,0.783990
400,0.492400,0.501123,0.869732,0.783990
500,0.344100,0.519308,0.851867,0.770648
600,0.353400,0.553697,0.870105,0.787166
700,0.329900,0.519374,0.878931,0.804320


TrainOutput(global_step=788, training_loss=0.4357920201296734, metrics={'train_runtime': 137.9811, 'train_samples_per_second': 91.244, 'train_steps_per_second': 5.711, 'total_flos': 188354626755840.0, 'train_loss': 0.4357920201296734, 'epoch': 2.0})

In [ ]:
test_true = torch.from_numpy(test_df['acceptable'].values).to(device)
with torch.no_grad():
  tokens = tokenizer(
      test_df['sentence'].to_list(),
      return_tensors='pt',
      padding=True
  ).to(device)
  preds = model(**tokens).logits.argmax(axis=1)

clf_metrics.compute(predictions=preds, references=test_true)

{'f1': 0.8712363869314542, 'accuracy': 0.7955239064089522}

In [ ]:
text0 = test_df[test_df['acceptable'] == 0].iloc[0, 0]
text0

'У многих туристов, кто посещают Кемер весной, есть шанс застать снег на вершине горы Тахталы и даже сочетать пляжный отдых с горнолыжным.'

In [ ]:
with torch.no_grad():
  tokens = tokenizer(
      text0,
      return_tensors='pt',
      padding=True
  ).to(device)
  label = model(**tokens).logits.argmax(axis=1)
print(f'Acceptable: {label.item()}')

Acceptable: 0


F1: 0.871\
Accuracy: 0.796 (не многим лучше, чем предсказание самого частотного класса - 0.745)

# 2. GPT

In [6]:
gpt_name = 'ai-forever/rugpt3large_based_on_gpt2'
tokenizer = AutoTokenizer.from_pretrained(gpt_name)
model = AutoModelForCausalLM.from_pretrained(gpt_name).to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

In [7]:
def get_acceptance(var_1, var_0, tokenizer=tokenizer, model=model):
  tokens_1 = tokenizer.encode(var_1, return_tensors='pt').to(device)
  tokens_0 = tokenizer.encode(var_0, return_tensors='pt').to(device)

  with torch.no_grad():
    loss_1 = model(tokens_1, labels=tokens_1).loss.item()
    loss_0 = model(tokens_0, labels=tokens_0).loss.item()

  if loss_1 <= loss_0:
    return 1
  else:
    return 0

## 2.1. Zero-shot

In [ ]:
prompts = ['Текст далее корректный? ', 'Текст далее приемлемый? ', 'Текст далее естественный? ']
refs = test_df['acceptable'].values
metrics = []

for pr in prompts:
  texts = [pr + sent for sent in test_df['sentence']]
  acc_1 = [sent + ' Ответ: да.' for sent in texts]
  acc_0 = [sent + ' Ответ: нет.' for sent in texts]

  scores = [get_acceptance(x_1, x_0) for x_1, x_0 in zip(acc_1, acc_0)]
  metrics.append(clf_metrics.compute(predictions=np.array(scores), references=refs))

In [ ]:
result = pd.DataFrame(index=prompts, data=metrics)
with pd.option_context('display.float_format', '{:.3f}'.format):
  print(result)

                              f1  accuracy
Текст далее корректный?    0.182     0.295
Текст далее приемлемый?    0.455     0.400
Текст далее естественный?  0.182     0.296


                               f1 | accuracy

---

Текст далее корректный? | 0.182 | 0.295\
Текст далее приемлемый? | 0.455 | 0.400\
Текст далее естественный? | 0.182 | 0.296

Наилучший промпт: "Текст далее приемлемый?"

## 2.2. Few-shot

In [34]:
prompt = 'Текст далее приемлемый? '
ex_1 = train_df[train_df['acceptable'] == 1].sample(2, random_state=945)
ex_0 = train_df[train_df['acceptable'] == 0].sample(2, random_state=945)
examples = []

for e0, e1 in zip(ex_0['sentence'], ex_1['sentence']):
  examples.append(f'{prompt}{e0} Ответ: нет.\n')
  examples.append(f'{prompt}{e1} Ответ: да.\n')

In [38]:
refs = test_df['acceptable'].values
metrics = []

for examp in accumulate(examples):
  texts = [f'{examp}{prompt}{sent}' for sent in test_df['sentence']]
  acc_1 = [sent + ' Ответ: да.' for sent in texts]
  acc_0 = [sent + ' Ответ: нет.' for sent in texts]

  scores = [get_acceptance(x_1, x_0) for x_1, x_0 in zip(acc_1, acc_0)]
  metrics.append(clf_metrics.compute(predictions=np.array(scores), references=refs))

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [40]:
result = pd.DataFrame(index=range(1, 5), data=metrics)
with pd.option_context('display.float_format', '{:.3f}'.format):
  print(result)

     f1  accuracy
1 0.276     0.327
2 0.721     0.587
3 0.568     0.462
4 0.552     0.456


                f1 | accuracy

---

1 example | 0.276 | 0.327\
2 examples | 0.721 | 0.587\
3 examples | 0.568 | 0.462\
4 examples | 0.552 | 0.456

Наилучший вариант получился, если привести только два примера.

#